In [20]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from src.preprocessing import train_valid_test_split
from src.evaluate import precision_k

In [36]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
train, valid, test = train_valid_test_split(df)
k = int(len(valid) * 0.1)

In [37]:
train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
train['pdays'] = train['pdays'].replace(-1, np.nan)

valid['pdays_prev_camp'] = valid['pdays'].ne(-1).astype('int8')
valid['pdays'] = valid['pdays'].replace(-1, np.nan)

/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_88780/759346521.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_88780/759346521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays'] = train['pdays'].replace(-1, np.nan)
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_88780/759346521.py:4: SettingWithCopyWarning: 
A value is trying to be set on a cop

In [38]:
X_train = train.drop(columns=['y', 'duration'])
y_train = train["y"]
X_valid = valid.drop(columns=['y', 'duration'])
y_valid = valid['y']

In [39]:

cat_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
)
cat_indices = [
      i for i, col in enumerate(X_train.columns)
      if X_train[col].dtype.name in ("object", "string", "category")
        ]

# eval_pool = Pool(X_valid, y_valid, cat_features=cat_indices)

In [40]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.01,
    loss_function="Logloss",
    depth=6,
    random_seed=0
)
model.fit(X_train, y_train, cat_features=cat_indices)

0:	learn: 0.6807925	total: 27.1ms	remaining: 13.5s
1:	learn: 0.6688290	total: 61.6ms	remaining: 15.3s
2:	learn: 0.6572036	total: 88ms	remaining: 14.6s
3:	learn: 0.6458782	total: 114ms	remaining: 14.1s
4:	learn: 0.6347666	total: 136ms	remaining: 13.5s
5:	learn: 0.6239129	total: 154ms	remaining: 12.7s
6:	learn: 0.6133812	total: 170ms	remaining: 11.9s
7:	learn: 0.6030065	total: 196ms	remaining: 12.1s
8:	learn: 0.5930211	total: 228ms	remaining: 12.4s
9:	learn: 0.5833602	total: 261ms	remaining: 12.8s
10:	learn: 0.5739682	total: 288ms	remaining: 12.8s
11:	learn: 0.5647090	total: 312ms	remaining: 12.7s
12:	learn: 0.5557684	total: 342ms	remaining: 12.8s
13:	learn: 0.5469953	total: 365ms	remaining: 12.7s
14:	learn: 0.5384142	total: 388ms	remaining: 12.5s
15:	learn: 0.5301554	total: 412ms	remaining: 12.5s
16:	learn: 0.5221151	total: 438ms	remaining: 12.4s
17:	learn: 0.5143396	total: 464ms	remaining: 12.4s
18:	learn: 0.5067565	total: 492ms	remaining: 12.4s
19:	learn: 0.4992616	total: 515ms	remain

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.01, loss_function='Logloss', random_seed=0)

In [31]:

y_pred = model.predict(X_valid)

In [41]:
yes_idx = list(model.classes_).index("yes")
y_score = model.predict_proba(X_valid)[:, yes_idx]
prec_k = precision_k(y_valid, y_score, k=k)
avg_precision = average_precision_score(y_valid, y_score, pos_label="yes")
# avg_precision = model.eval_metrics(data=eval_pool, metrics='Precision')

In [42]:
pd.Series({
    "precision_at_10pct": prec_k,
    "average_precision": avg_precision,
})

precision_at_10pct    0.293510
average_precision     0.236166
dtype: float64

#### exp 1:
- depth 6, iteration 500, random seed = 0, logloss, lr = 0.01
- metrics: prec_k = 0.2743, avg_prec = 0.2322

#### exp 2:
- same model pars
- metrics: prec_k = 0.2935, avg_prec = 0.2361